<a href="https://colab.research.google.com/github/berinbalci/BGT208-Yapay-Zeka-Siber-Guvenlik/blob/main/Hafta-04/BGT208_Hafta04_Makine_Ogrenmesinin_Temelleri.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# BGT208 – Yapay Zekâ ve Siber Güvenlik

## Hafta 4 – Makine Öğrenmesinin Temelleri

### Uygulamanın Amacı

Bu uygulamada makine öğrenmesinin temel çalışma süreci
PhiUSIIL Phishing URL Dataset üzerinden incelenecektir.

Uygulamada;

- özellik (feature) ve hedef (label) ayrımı,
- X ve y değişkenlerinin oluşturulması,
- eğitim ve test verisinin ayrılması,
- sınıf dağılımının korunması,
- model eğitimi (`fit()`),
- tahmin üretme (`predict()`)

işlemleri gerçekleştirilecektir.

> Bu haftanın amacı farklı sınıflandırma algoritmalarını karşılaştırmak değildir.
> Amaç, makine öğrenmesi sürecinin temel mantığını anlamaktır.

## 1. Gerekli Kütüphanelerin Eklenmesi

Bu uygulamada veri işlemleri için **Pandas**,
eğitim ve test verisini ayırmak için ise **Scikit-learn**
kütüphanesi kullanılacaktır.

In [ ]:
import pandas as pd

from sklearn.model_selection import train_test_split

## 2. Veri Setine Google Drive Üzerinden Erişim

Önceki haftalarda kullandığımız PhiUSIIL veri seti
Google Drive üzerinden Colab ortamına alınacaktır.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
dosya_yolu = "/content/drive/MyDrive/BGT208/PhiUSIIL_Phishing_URL_Dataset.csv"

df = pd.read_csv(dosya_yolu)

print("Veri seti yüklendi.")

In [ ]:
df.head()

In [ ]:
df.shape

## 3. Özellik (Feature) ve Hedef (Target) Değişkeninin Belirlenmesi

Denetimli makine öğrenmesinde veri setini iki temel bölüme ayırırız:

- **X (Features):** Modelin tahmin yapmak için kullanacağı bilgiler
- **y (Target / Label):** Modelin tahmin etmeye çalışacağı sonuç

PhiUSIIL veri setinde hedef değişken `label` sütunudur.

Ancak veri setindeki her sütunun doğrudan modele verilmesi uygun olmayabilir.
Örneğin metinsel veya yalnızca kayıtları tanımlamak amacıyla kullanılan
sütunların ayrıca değerlendirilmesi gerekir.

Bu nedenle önce veri setindeki sütunları ve veri tiplerini inceleyelim.

In [ ]:
df.dtypes

In [ ]:
df.select_dtypes(include="object").columns

### Sayısal ve Metinsel Değişkenler

Makine öğrenmesi algoritmalarının büyük bölümü doğrudan sayısal veriler
üzerinde çalışır.

Bu nedenle veri setindeki metinsel (`object`) sütunları doğrudan modele
vermeden önce nasıl kullanılacaklarına karar verilmesi gerekir.

Bu haftanın amacı veri kodlama yöntemlerini öğrenmek olmadığı için
uygulamada yalnızca **sayısal özellikleri** kullanacağız.

Hedef değişken olan `label` ise X içerisinden çıkarılarak ayrı bir
`y` değişkeninde tutulacaktır.

In [ ]:
X = df.select_dtypes(include="number").drop(columns=["label"])
y = df["label"]

In [ ]:
print("X boyutu:", X.shape)
print("y boyutu:", y.shape)

In [ ]:
X.head()

In [ ]:
y.head()

In [ ]:
### Ne Yaptık?

`X`, modelin phishing tespiti yaparken kullanacağı sayısal özellikleri
içermektedir.

`y` ise her kaydın gerçek sınıfını gösteren `label` değişkenidir.

Böylece veri setimizi:

**Özellikler (X) → Tahmin için kullanılan bilgiler**

ve

**Hedef (y) → Tahmin edilmek istenen sonuç**

şeklinde iki bölüme ayırmış olduk.

Henüz model eğitmedik. Bir sonraki adımda X ve y verilerini
**eğitim ve test verisi** olarak ayıracağız.

## 4. Verinin Eğitim ve Test Olarak Ayrılması

Bir makine öğrenmesi modelini geliştirirken veri setinin tamamını
modelin eğitiminde kullanmak doğru değildir.

Modelin daha önce görmediği veriler üzerinde nasıl çalıştığını
inceleyebilmek için veri setini iki bölüme ayırırız:

- **Eğitim verisi (Training Data):** Modelin öğrenmesi için kullanılır.
- **Test verisi (Test Data):** Eğitilen modelin daha önce görmediği
  örnekler üzerindeki davranışını incelemek için kullanılır.

Bu uygulamada verinin:

- %80'i eğitim,
- %20'si test

olarak ayrılacaktır.

Bunun için Scikit-learn kütüphanesindeki `train_test_split()`
fonksiyonunu kullanacağız.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

### Eğitim ve Test Verilerinin Boyutları

Veri setini ayırdıktan sonra oluşan eğitim ve test gruplarının
boyutlarını kontrol edelim.

`X_train` ve `y_train` modelin eğitiminde,
`X_test` ve `y_test` ise modelin daha önce görmediği örneklerde
kullanılacaktır.

In [ ]:
print("Tüm veri:", X.shape)

print("\nEğitim verisi:")
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("\nTest verisi:")
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

### `random_state=42` Ne İşe Yarar?

Veriler eğitim ve test gruplarına rastgele ayrılır.

`random_state` değerinin sabitlenmesi, kod tekrar çalıştırıldığında
aynı kayıtların eğitim ve test gruplarına ayrılmasını sağlar.

Bu özellik çalışmanın tekrar edilebilir olması açısından önemlidir.

`42` zorunlu veya özel bir sayı değildir. Sabit bir değer kullanılması
önemlidir.


### `stratify=y` Ne İşe Yarar?

`stratify=y`, hedef değişkendeki sınıf oranlarının eğitim ve test
verilerinde yaklaşık olarak korunmasını sağlar.

Örneğin tüm veri setinde phishing ve legitimate örneklerin belirli
bir dağılımı varsa, eğitim ve test gruplarında da benzer bir
dağılım oluşturulmaya çalışılır.

In [ ]:
print("Tüm veri:")
print(y.value_counts(normalize=True) * 100)

print("\nEğitim verisi:")
print(y_train.value_counts(normalize=True) * 100)

print("\nTest verisi:")
print(y_test.value_counts(normalize=True) * 100)

## 5. Basit Bir Makine Öğrenmesi Modelinin Eğitilmesi

Verimizi eğitim ve test olarak ayırdık.

Şimdi makine öğrenmesi sürecinin nasıl çalıştığını görmek için
basit bir sınıflandırma modeli oluşturacağız.

Bu uygulamada **Decision Tree (Karar Ağacı)** modeli kullanılacaktır.

> Bu hafta Karar Ağacı algoritmasının ayrıntılarına girmeyeceğiz.
> Buradaki amacımız bir makine öğrenmesi modelinin nasıl oluşturulduğunu,
> eğitildiğini ve tahmin yaptığını görmektir.

Scikit-learn'de temel süreç:

**Model oluştur → Eğit (`fit`) → Tahmin yap (`predict`)**

şeklindedir.

In [ ]:
from sklearn.tree import DecisionTreeClassifier

model = DecisionTreeClassifier(
    max_depth=5,
    random_state=42
)

### Model Oluşturuldu

`DecisionTreeClassifier()` ile bir sınıflandırma modeli oluşturduk.

Model şu anda henüz hiçbir şey öğrenmedi.

Modelin veri setindeki örüntüleri öğrenebilmesi için eğitim verilerini
kullanmamız gerekir.

`max_depth=5`, ağacın aşırı büyümesini sınırlamak için kullanılmıştır.
Bu parametrenin model üzerindeki etkisi ilerleyen haftalarda ele alınacaktır.

In [ ]:
model.fit(X_train, y_train)

### `fit()` Ne Yaptı?

`fit()` modelin **eğitim aşamasıdır**.

Model;

- `X_train` içerisindeki özellikleri,
- `y_train` içerisindeki gerçek sınıfları

birlikte kullanarak verideki örüntüleri öğrenmeye çalışır.

Model eğitim sırasında **test verisini görmez**.

## 6. Model ile Tahmin Yapılması

Model eğitimini tamamladıktan sonra daha önce görmediği test
örnekleri üzerinde tahmin yapabilir.

Scikit-learn'de tahmin yapmak için `predict()` fonksiyonu kullanılır.

Model `X_test` içerisindeki özellikleri görecek ancak gerçek sınıfları
tahmin sırasında kullanmayacaktır.

In [ ]:
y_pred = model.predict(X_test)

In [ ]:
print("Modelin ilk 10 tahmini:")
print(y_pred[:10])

In [ ]:
print("Gerçek sınıflar:")
print(y_test.iloc[:10].values)

print("\nModelin tahminleri:")
print(y_pred[:10])

### Gerçek Değer ve Tahmin

Burada iki farklı bilgi bulunmaktadır:

- `y_test` → örneklerin gerçek sınıfları
- `y_pred` → modelin tahmin ettiği sınıflar

Bazı tahminlerin gerçek değerlerle aynı, bazılarının ise farklı
olabileceğini görebiliriz.

Modelin ne kadar başarılı olduğunu yalnızca birkaç örneğe bakarak
değerlendirmek doğru değildir.

Bir modelin performansını değerlendirmek için farklı ölçütler kullanılır.

**Accuracy, Precision, Recall, F1-score ve Confusion Matrix**
gibi performans ölçütleri ilerleyen haftalarda ayrıntılı olarak
ele alınacaktır.

# Uygulama Sonucu

Bu uygulamada bir makine öğrenmesi çalışmasının temel adımları
PhiUSIIL Phishing URL Dataset üzerinden incelenmiştir.

Uygulama kapsamında:

- Veri seti Google Drive üzerinden çalışma ortamına alındı.
- Modelde kullanılacak özellikler (**X**) ve hedef değişken (**y**) ayrıldı.
- Veri seti eğitim ve test gruplarına bölündü.
- `random_state` ve `stratify` kavramları incelendi.
- Basit bir Karar Ağacı modeli oluşturuldu.
- `fit()` ile model eğitim verileri üzerinde eğitildi.
- `predict()` ile daha önce görülmeyen test verileri için tahminler üretildi.
- Gerçek sınıflar ile model tahminlerinin farklı kavramlar olduğu görüldü.

## Makine Öğrenmesi Akışı

Bu haftaki uygulamanın temel süreci:

**Veri → X ve y → Eğitim/Test Ayrımı → Model → fit() → predict()**

şeklinde özetlenebilir.

Bu aşamada modelin ne kadar başarılı olduğu değerlendirilmemiştir.

---

## Sonraki Hafta

Bir sonraki hafta **Sınıflandırma Modelleri** ele alınacaktır.

Aynı siber güvenlik problemi üzerinde farklı sınıflandırma
algoritmaları uygulanarak modellerin çalışma biçimleri incelenecektir.